In [37]:
%matplotlib

import torch
import numpy as np
import matplotlib.pyplot as plt

Using matplotlib backend: TkAgg


In [2]:
trunk_x_0 = torch.tensor([[0, 0, 0.3]])
trunk_xd_0 = torch.zeros_like(trunk_x_0)
trunk_o_0 = torch.zeros_like(trunk_x_0)
trunk_od_0 = torch.zeros_like(trunk_x_0)

In [3]:
trunk_x_lo = torch.tensor([[-0.0150,  0.0809,  0.3130]])
trunk_xd_lo = torch.tensor([[-0.0969,  0.5238,  0.9596]])
trunk_o_lo = torch.tensor([[-0.1285, -0.0310, -0.0767]])
trunk_od_lo = torch.tensor([[0.9700, -0.1732, -0.9753]])
t_th = torch.tensor([[0.4077,]])

In [4]:
def BezierTrajectory(start: torch.Tensor, start_v: torch.Tensor, end: torch.Tensor, end_v: torch.Tensor, t_th: torch.Tensor, t_ex: float):

    w = torch.stack((
        start.unsqueeze(1),
        (((t_th / 3) * start_v) + start).unsqueeze(1),
        ((-(t_th / 3) * end_v) + end).unsqueeze(1),
        end.unsqueeze(1)), dim=2).squeeze()

    w = w.reshape(-1, 4, 3)

    w_d = torch.stack((
        ((3 / t_th) * (w[:, 1] - w[:, 0])).unsqueeze(1),
        ((3 / t_th) * (w[:, 2] - w[:, 1])).unsqueeze(1),
        ((3 / t_th) * (w[:, 3] - w[:, 2])).unsqueeze(1)), dim=2).squeeze()

    t = t_ex / t_th

    bezier_curve_3 = torch.cat((
        (1.0) * (t**0) * (1 - t)**(3 - 0),
        (3.0) * (t**1) * (1 - t)**(3 - 1),
        (3.0) * (t**2) * (1 - t)**(3 - 2),
        (1.0) * (t**3) * (1 - t)**(3 - 3),
    ), dim=-1)

    bezier_curve_2 = torch.cat((
        (1.0) * (t**0) * (1 - t)**(2 - 0),
        (2.0) * (t**1) * (1 - t)**(2 - 1),
        (1.0) * (t**2) * (1 - t)**(2 - 2),
    ), dim=-1)

    bezier_position = torch.cat((
        (w[..., 0] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
        (w[..., 1] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
        (w[..., 2] * bezier_curve_3).sum(dim=-1).reshape(-1, 1)), dim=1)

    bezier_velocity = torch.cat((
        (w_d[..., 0] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
        (w_d[..., 1] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
        (w_d[..., 2] * bezier_curve_2).sum(dim=-1).reshape(-1, 1)), dim=1)

    return bezier_position, bezier_velocity

In [5]:
max_time = t_th.item() + 0.005

In [6]:
pos_lin_vel = [BezierTrajectory(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
pos = torch.stack([val[0] for val in pos_lin_vel], dim=1)
lin_vel = torch.stack([val[1] for val in pos_lin_vel], dim=1)

orient_ang_vel = [BezierTrajectory(trunk_o_0, trunk_od_0, trunk_o_lo, trunk_od_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
orient = torch.stack([val[0] for val in orient_ang_vel], dim=1)
ang_vel = torch.stack([val[1] for val in orient_ang_vel], dim=1)

In [7]:
# plt.close()

fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

i = 0
step_size = 5
draw_l_vel = False
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
if draw_l_vel:
    ax.quiver(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2],
              lin_vel[i][..., 0], lin_vel[i][..., 1], lin_vel[i][..., 2],
              length=0.1, color='purple', arrow_length_ratio=0)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

for j in range(0, len(orient[i]), step_size):
    # Getting the current position and orientation
    p = pos[i, j].numpy()
    o = orient[i, j].numpy()

    # Define the Cartesian frame vectors (x, y, z)
    x_vec = np.array([1, 0, 0])  # X-axis
    y_vec = np.array([0, 1, 0])  # Y-axis
    z_vec = np.array([0, 0, 1])  # Z-axis

    # Rotate the frame based on orientation
    rotation_matrix = np.eye(3)
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [1, 0, 0],
        [0, np.cos(o[0]), -np.sin(o[0])],
        [0, np.sin(o[0]), np.cos(o[0])]
    ]))
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [np.cos(o[1]), 0, np.sin(o[1])],
        [0, 1, 0],
        [-np.sin(o[1]), 0, np.cos(o[1])]
    ]))
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [np.cos(o[2]), -np.sin(o[2]), 0],
        [np.sin(o[2]), np.cos(o[2]), 0],
        [0, 0, 1]
    ]))

    x_vec = np.matmul(rotation_matrix, x_vec)
    y_vec = np.matmul(rotation_matrix, y_vec)
    z_vec = np.matmul(rotation_matrix, z_vec)

    # Plotting the frame
    ax.quiver(p[0], p[1], p[2], x_vec[0], x_vec[1], x_vec[2], length=0.05, color='r', arrow_length_ratio=0)
    ax.quiver(p[0], p[1], p[2], y_vec[0], y_vec[1], y_vec[2], length=0.05, color='g', arrow_length_ratio=0)
    ax.quiver(p[0], p[1], p[2], z_vec[0], z_vec[1], z_vec[2], length=0.05, color='b', arrow_length_ratio=0)

ax.quiver(pos[...,-1, 0], pos[...,-1, 1], pos[...,-1, 2], lin_vel[...,-1, 0], lin_vel[...,-1, 1], lin_vel[...,-1, 2], length=0.1, normalize=False, color='purple', arrow_length_ratio=0.1)

plt.show()

Adding velocity

In [8]:
v0 = trunk_xd_lo
v_mult = torch.tensor([2.6280])
vf = v0 * v_mult

In [9]:
vf

tensor([[-0.2547,  1.3765,  2.5218]])

In [10]:
v_un = v0 / torch.norm(v0)
v_un

tensor([[-0.0883,  0.4773,  0.8743]])

In [11]:
s0 = trunk_x_lo

In [12]:
l_exp = torch.tensor([0.1397])

In [13]:
sf = s0 + v_un * l_exp.reshape(-1,1)

In [14]:
sf

tensor([[-0.0273,  0.1476,  0.4351]])

In [15]:
vf_n = torch.norm(vf)
v0_n = torch.norm(v0)
sf_n = torch.norm(sf)
s0_n = torch.norm(s0)

In [16]:
a = 0.5 * ((np.power(vf_n, 2)-np.power(v0_n, 2))/(sf_n-s0_n))

In [17]:
a, a/9.81

(tensor(26.0302, dtype=torch.float64), tensor(2.6534, dtype=torch.float64))

In [24]:
t_exp = torch.tensor([(vf_n-v0_n)/a])

In [25]:
t_exp

tensor([0.0686], dtype=torch.float64)

In [26]:
t_th+ t_exp

tensor([[0.4763]], dtype=torch.float64)

In [28]:
sim_time = torch.arange(0,t_exp.item(),0.005)

In [29]:
sim_time

tensor([0.0000, 0.0050, 0.0100, 0.0150, 0.0200, 0.0250, 0.0300, 0.0350, 0.0400,
        0.0450, 0.0500, 0.0550, 0.0600, 0.0650])

In [31]:
[t/t_exp for t in sim_time]

[tensor([0.], dtype=torch.float64),
 tensor([0.0728], dtype=torch.float64),
 tensor([0.1457], dtype=torch.float64),
 tensor([0.2185], dtype=torch.float64),
 tensor([0.2914], dtype=torch.float64),
 tensor([0.3642], dtype=torch.float64),
 tensor([0.4370], dtype=torch.float64),
 tensor([0.5099], dtype=torch.float64),
 tensor([0.5827], dtype=torch.float64),
 tensor([0.6556], dtype=torch.float64),
 tensor([0.7284], dtype=torch.float64),
 tensor([0.8012], dtype=torch.float64),
 tensor([0.8741], dtype=torch.float64),
 tensor([0.9469], dtype=torch.float64)]

In [34]:
s = torch.stack([torch.lerp(torch.tensor(s0), torch.tensor(sf), (t/t_exp).to(torch.float)) for t in sim_time])
s

/tmp/ipykernel_2887602/460731555.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  s = torch.stack([torch.lerp(torch.tensor(s0), torch.tensor(sf), (t/t_exp).to(torch.float)) for t in sim_time])


tensor([[[-0.0150,  0.0809,  0.3130]],

        [[-0.0159,  0.0858,  0.3219]],

        [[-0.0168,  0.0906,  0.3308]],

        [[-0.0177,  0.0955,  0.3397]],

        [[-0.0186,  0.1003,  0.3486]],

        [[-0.0195,  0.1052,  0.3575]],

        [[-0.0204,  0.1100,  0.3664]],

        [[-0.0213,  0.1149,  0.3753]],

        [[-0.0222,  0.1198,  0.3842]],

        [[-0.0231,  0.1246,  0.3931]],

        [[-0.0240,  0.1295,  0.4020]],

        [[-0.0249,  0.1343,  0.4109]],

        [[-0.0258,  0.1392,  0.4198]],

        [[-0.0267,  0.1440,  0.4287]]])

In [35]:
v = torch.stack([torch.lerp(torch.tensor(v0), torch.tensor(vf),(t/t_exp).to(torch.float)) for t in sim_time if t <= t_exp])
v

/tmp/ipykernel_2887602/3212860210.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  v = torch.stack([torch.lerp(torch.tensor(v0), torch.tensor(vf),(t/t_exp).to(torch.float)) for t in sim_time if t <= t_exp])


tensor([[[-0.0969,  0.5238,  0.9596]],

        [[-0.1084,  0.5859,  1.0734]],

        [[-0.1199,  0.6480,  1.1872]],

        [[-0.1314,  0.7101,  1.3010]],

        [[-0.1429,  0.7723,  1.4148]],

        [[-0.1544,  0.8344,  1.5286]],

        [[-0.1658,  0.8965,  1.6424]],

        [[-0.1773,  0.9586,  1.7562]],

        [[-0.1888,  1.0207,  1.8699]],

        [[-0.2003,  1.0828,  1.9837]],

        [[-0.2118,  1.1449,  2.0975]],

        [[-0.2233,  1.2071,  2.2113]],

        [[-0.2348,  1.2692,  2.3251]],

        [[-0.2463,  1.3313,  2.4389]]])

In [38]:
# plt.close()

fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

i = 0
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

ax.scatter(s[..., 0], s[..., 1], s[..., 2], color='black', linewidths=1)
ax.scatter(sf[..., 0], sf[..., 1], sf[..., 2], alpha=1, linewidths=5)
ax.quiver(pos[..., -1, 0], pos[..., -1, 1], pos[..., -1, 2], lin_vel[..., -1, 0], lin_vel[..., -1, 1], lin_vel[..., -1, 2], length=0.1, normalize=False, color='orange', arrow_length_ratio=0.1)
ax.quiver(s[..., 0][-1], s[..., 1][-1], s[..., 2][-1], v[..., 0][-1], v[..., 1][-1], v[..., 2][-1], length=0.1, normalize=False, color='green', arrow_length_ratio=0.1)


plt.show()

In [31]:
trunk_x_lo = torch.tensor([[0,0,0.35]])
trunk_xd_lo = torch.tensor([[0,0,2.5]])
t_th = torch.tensor([[0.5500]])

In [32]:
max_time = 0.5500 + 0.005

In [33]:
pos_lin_vel = [BezierTrajectory(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
pos = torch.stack([val[0] for val in pos_lin_vel], dim=1)
lin_vel = torch.stack([val[1] for val in pos_lin_vel], dim=1)

In [34]:
# plt.close()

fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

i = 0
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

ax.quiver(pos[...,-1, 0], pos[...,-1, 1], pos[...,-1, 2], lin_vel[...,-1, 0], lin_vel[...,-1, 1], lin_vel[...,-1, 2], length=0.1, normalize=False, color='orange', arrow_length_ratio=0.1)


plt.show()